In [0]:
# ============================================================
# SILVER ORDERS PIPELINE
# Incremental Processing + DQ + Dedup + MERGE + Watermark
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql.functions import (
    col,
    to_timestamp,
    trim,
    initcap,
    row_number,
    concat_ws,
    when,
    lit,
    current_timestamp,
    max
)

from pyspark.sql.window import Window

from delta.tables import DeltaTable

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

import uuid

In [0]:
# ============================================================
# 2. PATHS
# ============================================================

bronze_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "bronze/orders/"
)

silver_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/orders/"
)

quarantine_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "quarantine/orders/"
)

control_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)

pipeline_name = "orders_silver"

In [0]:
# ============================================================
# 3. CREATE RUN ID
# ============================================================
run_id = "RUN_" + str(uuid.uuid4())[:8]
print( run_id)


RUN_f15430a5


In [0]:
# ============================================================
# 4. READ BRONZE
# ============================================================

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
bronze_count = bronze_df.count()
print("Bronze records:", bronze_count)

Bronze records: 125003


In [0]:
# ============================================================
# 5. CREATE SHARED CONTROL TABLE IF NEEDED
# ============================================================

control_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])
if not DeltaTable.isDeltaTable(
    spark,
    control_path
):
    empty_control_df = spark.createDataFrame(
        [],
        control_schema
    )
    (
        empty_control_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(control_path)
    )

    print("Shared control table created.")

In [0]:
# ============================================================
# 6. INITIALIZE ORDERS PIPELINE ROW
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)
orders_control_count = (
    control_df
    .filter(
        col("pipeline_name") == pipeline_name
    )
    .count()
)
if orders_control_count == 0:
    initial_orders_control_df = (
        spark.createDataFrame(
            [
                (
                    pipeline_name,
                    None,
                    None,
                    None
                )
            ],
            control_schema
        )
    )
    (
        initial_orders_control_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )
    print(
        "orders_silver initialized in control table."
    )
else:
    print(
        "orders_silver already exists in control table."
    )

orders_silver already exists in control table.


In [0]:
# ============================================================
# 7. READ ORDERS WATERMARK
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)
orders_control = (
    control_df
    .filter(
        col("pipeline_name") == pipeline_name
    )
)
last_processed_timestamp = (
    orders_control
    .select("last_processed_timestamp")
    .collect()[0][0]
)
print(
    "Last processed timestamp:",
    last_processed_timestamp
)
control_df.printSchema()

Last processed timestamp: 2026-10-09 02:49:13
root
 |-- pipeline_name: string (nullable = true)
 |-- last_processed_timestamp: timestamp (nullable = true)
 |-- last_run_id: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [0]:
bronze_transformed = (
    bronze_df
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("sales_amount", col("sales_amount").cast("double"))
    .withColumn(
        "order_timestamp",
        col("order_timestamp").cast("timestamp")
    )
    .withColumn(
        "updated_at",
        col("updated_at").cast("timestamp")
    )
    .withColumn(
        "market",
        initcap(trim(col("market")))
    )
)

In [0]:
# ============================================================
# 9. INCREMENTAL FILTER
# ============================================================

bronze_transformed.printSchema()

if last_processed_timestamp is None:

    print("First Orders Silver run.")
    incremental_df = bronze_transformed

else:

    print("Incremental Orders Silver run.")

    incremental_df = bronze_transformed.filter(
        col("updated_at") > lit(last_processed_timestamp)
    )

incremental_count = incremental_df.count()

print("Incremental records:", incremental_count)

root
 |-- customer_id: long (nullable = true)
 |-- market: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- batch_id: string (nullable = true)

Incremental Orders Silver run.
Incremental records: 50003


In [0]:
# ============================================================
# 10. STOP IF NOTHING NEW
# ============================================================

if incremental_count == 0:
    print("No new records to process.")
    dbutils.notebook.exit("SUCCESS: No new records")

In [0]:
bronze_transformed.select(
    "order_timestamp",
    "updated_at"
).show(10)

+-------------------+-------------------+
|    order_timestamp|         updated_at|
+-------------------+-------------------+
|2026-10-10 08:00:00|2026-10-10 08:00:00|
|2026-10-10 08:00:02|2026-10-10 08:00:02|
|2026-10-10 08:00:04|2026-10-10 08:00:04|
|2026-10-10 08:00:06|2026-10-10 08:00:06|
|2026-10-10 08:00:08|2026-10-10 08:00:08|
|2026-10-10 08:00:10|2026-10-10 08:00:10|
|2026-10-10 08:00:12|2026-10-10 08:00:12|
|2026-10-10 08:00:14|2026-10-10 08:00:14|
|2026-10-10 08:00:16|2026-10-10 08:00:16|
|2026-10-10 08:00:18|2026-10-10 08:00:18|
+-------------------+-------------------+
only showing top 10 rows


In [0]:
# ========================================================
    # 11. EXACT DUPLICATE REMOVAL
    # ========================================================

incremental_df = (
        incremental_df
        .dropDuplicates()
    )

print(
        "Records after exact dedup:",
        incremental_df.count()
    )

Records after exact dedup: 50003


In [0]:
# ========================================================
    # 12. BUSINESS KEY DEDUPLICATION
    # ========================================================

window_spec = (
        Window
        .partitionBy("order_id")
        .orderBy(
            col("updated_at").desc()
        )
    )


silver_df = (
        incremental_df
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
        .filter(
            col("row_num") == 1
        )
        .drop("row_num")
    )


print(
        "Records after business-key dedup:",
        silver_df.count()
    )

Records after business-key dedup: 50003


In [0]:
   # ========================================================
    # 13. VALID MARKETS
    # ========================================================

valid_markets = [
        "India",
        "Canada",
        "UK",
        "Australia",
        "Germany"
    ]

In [0]:
    # ========================================================
    # 14. DATA QUALITY RULES
    # ========================================================

invalid_condition = (

        col("order_id").isNull()
        |
        col("customer_id").isNull()
        |
        col("product_id").isNull()
        |
        col("market").isNull()
        # |
        # ~col("market").isin(valid_markets)
        |
        col("quantity").isNull()
        |
        (col("quantity") <= 0)
        |
        col("sales_amount").isNull()
        |
        (col("sales_amount") <= 0)
        |
        col("order_timestamp").isNull()
        |
        col("updated_at").isNull()
    )

In [0]:
    # ========================================================
    # 15. SPLIT VALID / INVALID
    # ========================================================
invalid_df = (
        silver_df
        .filter(invalid_condition)
    )
df_valid = (
        silver_df
        .filter(~invalid_condition)
    )
valid_count = df_valid.count()
invalid_count = invalid_df.count()
print(
        "Valid records:",
        valid_count
    )
print(
        "Invalid records:",
        invalid_count
    )

Valid records: 50003
Invalid records: 0


In [0]:
    # ========================================================
    # 16. ADD REJECTION REASON
    # ========================================================

invalid_df = (
        invalid_df
        .withColumn(
            "rejection_reason",
            concat_ws(
                ";",
                when(
                    col("order_id").isNull(),
                    lit("MISSING_ORDER_ID")
                ),
                when(
                    col("customer_id").isNull(),
                    lit("MISSING_CUSTOMER_ID")
                ),
                when(
                    col("product_id").isNull(),
                    lit("MISSING_PRODUCT_ID")
                ),
                when(
                    col("market").isNull(),
                    lit("MISSING_MARKET")
                ),
                when(
                    col("market").isNotNull()
                    &
                    ~col("market").isin(valid_markets),
                    lit("INVALID_MARKET")
                ),
                when(
                    col("quantity").isNull(),
                    lit("MISSING_QUANTITY")
                ),
                when(
                    col("quantity") <= 0,
                    lit("INVALID_QUANTITY")
                ),
                when(
                    col("sales_amount").isNull(),
                    lit("MISSING_SALES_AMOUNT")
                ),
                when(
                    col("sales_amount") <= 0,
                    lit("INVALID_SALES_AMOUNT")
                ),
                when(
                    col("order_timestamp").isNull(),
                    lit("MISSING_ORDER_TIMESTAMP")
                ),
                when(
                    col("updated_at").isNull(),
                    lit("MISSING_UPDATED_AT")
                )
            )
        )
        .withColumn(
            "quarantine_timestamp",
            current_timestamp()
        )
        .withColumn(
            "quarantine_run_id",
            lit(run_id)
        )
    )

In [0]:

    # ========================================================
    # 17. WRITE VALID DATA → SILVER
    # ========================================================

if valid_count > 0:
    silver_exists = DeltaTable.isDeltaTable(
            spark,
            silver_path
        )
        # ====================================================
        # FIRST RUN → CREATE SILVER
        # ====================================================

    if not silver_exists:
        print(
                "Silver Orders table does not exist."
            )
        print(
                "Creating Silver Delta table..."
            )
        (
                df_valid
                .write
                .format("delta")
                .mode("overwrite")
                .save(silver_path)
            )
        print(
                "Silver Orders table created successfully."
            )
        # ====================================================
        # SUBSEQUENT RUN → MERGE
        # ====================================================

    else:
        print(
                "Merging valid records into Silver..."
            )
        silver_table = DeltaTable.forPath(
                spark,
                silver_path
            )
        (
    silver_table.alias("target")
    .merge(
        df_valid.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdate(
        condition="source.updated_at > target.updated_at",
        set={
            "customer_id": "source.customer_id",
            "product_id": "source.product_id",
            "market": "source.market",
            "quantity": "source.quantity",
            "sales_amount": "source.sales_amount",
            "order_timestamp": "source.order_timestamp",
            "updated_at": "source.updated_at",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "source_system": "source.source_system",
            "source_file": "source.source_file",
            "run_id": "source.run_id",
            "batch_id": "source.batch_id"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)


        print(
                "Valid records MERGED into Silver."
            )


else:

    print(
            "No valid records to write to Silver."
        )

Silver Orders table does not exist.
Creating Silver Delta table...
Silver Orders table created successfully.


In [0]:
    # ========================================================
    # 18. WRITE INVALID DATA → QUARANTINE
    # ========================================================

if invalid_count > 0:

    (
            invalid_df
            .write
            .format("delta")
            .mode("append")
            .option(
                "mergeSchema",
                "true"
            )
            .save(quarantine_path)
        )
    print(
            "Invalid records written to Quarantine."
        )
else:
    print(
            "No invalid records to quarantine."
        )

No invalid records to quarantine.


In [0]:
    # ========================================================
    # 19. CALCULATE NEW WATERMARK
    # ========================================================

new_watermark = (
        incremental_df
        .select(
            max("updated_at")
            .alias("max_updated_at")
        )
        .collect()[0]["max_updated_at"]
    )
print(
        "New watermark:",
        new_watermark
    )

New watermark: 2026-10-14 23:59:56


In [0]:
# ========================================================
    # 20. UPDATE SHARED CONTROL TABLE
    # ========================================================

watermark_update_schema = StructType([
        StructField(
            "pipeline_name",
            StringType(),
            False
        ),

        StructField(
            "last_processed_timestamp",
            TimestampType(),
            True
        ),

        StructField(
            "last_run_id",
            StringType(),
            True
        )
    ])


watermark_update_df = (
        spark.createDataFrame(
            [
                (
                    pipeline_name,
                    new_watermark,
                    run_id
                )
            ],
            watermark_update_schema
        )
        .withColumn(
            "updated_at",
            current_timestamp()
        )
    )


control_table = DeltaTable.forPath(
        spark,
        control_path
    )


(
        control_table.alias("target")

        .merge(
            watermark_update_df.alias("source"),

            """
            target.pipeline_name =
            source.pipeline_name
            """
        )

        .whenMatchedUpdate(
            set={

                "last_processed_timestamp":
                    "source.last_processed_timestamp",

                "last_run_id":
                    "source.last_run_id",

                "updated_at":
                    "source.updated_at"
            }
        )

        .whenNotMatchedInsert(
            values={

                "pipeline_name":
                    "source.pipeline_name",

                "last_processed_timestamp":
                    "source.last_processed_timestamp",

                "last_run_id":
                    "source.last_run_id",

                "updated_at":
                    "source.updated_at"
            }
        )

        .execute()
    )


print(
        "===================================="
    )

print(
        "Control table updated successfully."
    )

print(
        "====================================")

Control table updated successfully.


In [0]:
# ============================================================
# 21. FINAL CONTROL TABLE CHECK
# ============================================================

final_control_df = (
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        col("pipeline_name") == pipeline_name
    )
)

display(final_control_df)

pipeline_name,last_processed_timestamp,last_run_id,updated_at
orders_silver,2026-10-14T23:59:56.000Z,RUN_f15430a5,2026-10-14T13:52:22.202Z


In [0]:
# ============================================================
# 22. FINAL SILVER CHECK
# ============================================================

if DeltaTable.isDeltaTable(
    spark,
    silver_path
):

    final_silver_df = (
        spark.read
        .format("delta")
        .load(silver_path)
    )

    print(
        "Final Silver Orders records:",
        final_silver_df.count()
    )

    display(
        final_silver_df
        .orderBy("order_id")
    )

Final Silver Orders records: 50003


customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at,ingestion_timestamp,source_system,source_file,run_id,batch_id
140001,India,400001,2026-10-14T00:00:00.000Z,240001,1,182.29,2026-10-14T00:00:00.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140002,Us,400002,2026-10-14T00:01:37.000Z,240002,2,89.9,2026-10-14T00:01:37.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140003,Canada,400003,2026-10-14T00:03:14.000Z,240003,3,577.17,2026-10-14T00:03:14.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140004,Netherlands,400004,2026-10-14T00:04:51.000Z,240004,4,808.52,2026-10-14T00:04:51.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140005,Uk,400005,2026-10-14T00:06:28.000Z,240005,5,741.85,2026-10-14T00:06:28.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140006,India,400006,2026-10-14T00:08:05.000Z,240006,1,83.69,2026-10-14T00:08:05.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140007,Us,400007,2026-10-14T00:09:42.000Z,240007,2,180.88,2026-10-14T00:09:42.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140008,Canada,400008,2026-10-14T00:11:19.000Z,240008,3,396.81,2026-10-14T00:11:19.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140009,Netherlands,400009,2026-10-14T00:12:56.000Z,240009,4,797.92,2026-10-14T00:12:56.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
140010,Uk,400010,2026-10-14T00:14:33.000Z,240010,5,1168.2,2026-10-14T00:14:33.000Z,2026-10-14T08:35:21.740Z,orders_api,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=14/orders_20261014.json,2aaf1c40-a17f-4c3f-9501-921d7a1ed27e,4e80a8a5-c67f-490f-87cd-1ecce8227d78
